In [4]:
from pathlib import Path
import polars as pl

ricu_concept_dict = pl.read_json("/workspaces/example/data/projects/ricu/inst/extdata/config/concept-dict.json")

d_items_df = pl.scan_csv("/workspaces/example/data/physionet.org/files/mimiciv/3.1/icu/d_items.csv.gz")
d_items_df = d_items_df.with_columns((pl.col("itemid").cast(pl.Utf8) + "//" + pl.col("label")).alias("code"))

d_labitems_df = pl.scan_csv("/workspaces/example/data/physionet.org/files/mimiciv/3.1/hosp/d_labitems.csv.gz")
d_labitems_df = d_labitems_df.with_columns((pl.col("itemid").cast(pl.Utf8) + "//" + pl.col("label")).alias("code"))

In [ ]:
def extract_concept_config() -> tuple[dict, dict]:
    data_by_category = {}
    data_by_name = {}
    for concept in ricu_concept_dict.columns:
        if (sources := ricu_concept_dict[concept][0].get("sources")) and sources.get("miiv"):
            if (ids := (miiv := sources["miiv"][0]).get("ids")) is None:
                continue
            cnpt_entry = ricu_concept_dict[concept][0]
            cnpt_category = cnpt_entry["category"]
            name = cnpt_entry["description"].replace(" ", "_")
            table = miiv["table"]

            if not data_by_category.get(cnpt_category):
                data_by_category[cnpt_category] = {}

            if isinstance(ids, str) or (isinstance(ids, list) and isinstance(ids[0], str)):
                continue
            ids: list[str] = ids if isinstance(ids, list) else [ids]
            data_by_category[cnpt_category][name] = {
                "name": cnpt_entry["description"].replace(" ", "_"),
                "unit" : cnpt_entry.get("unit"),
                "table" : table,
                "code" : None,
                "ids" : ids,
                "short" : concept,
                "min" : cnpt_entry.get("min"),
                "max" : cnpt_entry.get("max"), 
            }
            
            filtered_items = (d_labitems_df if table == "labevents" else d_items_df).filter(pl.col("itemid").is_in(ids)).select("code").collect()
            code_list = list(filtered_items.to_dict()["code"])
            regex = "|".join(code_list)

            data_by_category[cnpt_category][name]["code"] = "(" + regex + ")"
            data_by_name[name] = data_by_category[cnpt_category][name]
            data_by_name["category"] = cnpt_category
    return (data_by_category, data_by_name)

In [ ]:
(_, data_by_name) = extract_concept_config()

# Root folder were the concepts are saved
cnpt_root_folder = Path("/workspaces/OpenICU-Example/output/project/workspace/concept")

n = extract_concept_config()

found_data_info = {}

# iterate all the folders in the root folder
for cnpt_folder in cnpt_root_folder.iterdir():
    # if the cnpt_folder is a folder/directory execute logic
    if cnpt_folder.is_dir():
        # select the first file in the sub folder "/1.0.0/" and exprect it to be the cnpt_file
        cnpt_file  = next((file for file in (cnpt_folder / "1.0.0").iterdir() if file.is_file()), None)
        # print(f"{cnpt_folder}: {cnpt_file.name if cnpt_file else "empty"}")
        size = len(pl.read_parquet(cnpt_file))
        # print(size)
        name = cnpt_folder.name
        if data_by_name.get(name):
            found_data_info[name] = data_by_name[name]
        else:
            print(name)

In [7]:
import rpy2.robjects as ro

def dict_to_r(d):
    items = {}
    for k, v in d.items():
        if isinstance(v, str):
            items[k] = ro.StrVector([v])
        elif isinstance(v, (int, float)):
            items[k] = ro.FloatVector([v])
        elif isinstance(v, list):
            items[k] = ro.FloatVector(v) if isinstance(v[0], (int, float)) else ro.StrVector(v)
        else:
            items[k] = ro.StrVector([str(v)])  # fallback
    return ro.ListVector(items)

In [8]:
import rpy2.robjects as ro
from rpy2.robjects import pandas2ri
from rpy2.robjects.conversion import localconverter

with localconverter(ro.default_converter + pandas2ri.converter):
    ro.globalenv["found_data_info"] = dict_to_r(found_data_info)

In [9]:
import json

with open("found_data_info.json", "w") as f:
    json.dump(found_data_info, f)

In [10]:
# %%R

# library(jsonlite)

# for (values in found_data_info){
#     cleaned <- gsub("'", '"', values)
#     cleaned <- gsub("None", 'null', cleaned)
#     parsed  <- fromJSON(cleaned)
#     print(parsed)
#     break
# }

In [ ]:
%%R
library(ricu)
library(jsonlite)

blacklist = c("dbp", "hr", "map")

get_free_ram_gb <- function() {
    mem <- gc(verbose=FALSE)
    free <- tryCatch({
        as.numeric(system("awk '/MemAvailable/ {print $2}' /proc/meminfo", intern=TRUE)) / 1024 / 1024
    }, error = function(e) NA)
    round(free, 2)
}

local_patient <- as.data.frame(miiv$patients[, c("subject_id", "anchor_year", "anchor_age")])
local_patient$birthdate <- as.POSIXct(
    paste0(local_patient$anchor_year - local_patient$anchor_age, "-01-01 00:00:00"),
    format = "%Y-%m-%d %H:%M:%S", tz = "UTC"
)

for (values in found_data_info) {
    cleaned <- gsub("'", '"', values)
    cleaned <- gsub("None", "null", cleaned)
    parsed  <- fromJSON(cleaned)
    if(parsed$short %in% blacklist){
        next
    }

    min_val <- if (!is.null(parsed$min)) as.numeric(parsed$min) else NULL
    max_val <- if (!is.null(parsed$max)) as.numeric(parsed$max) else NULL

    # RAM vor load_concepts
    cat(sprintf("[%s] Freier RAM vor load_concepts: %.2f GB\n", parsed$short, get_free_ram_gb()))
    cnpt <- load_concepts(parsed$short, src="miiv", aggregate=FALSE, interval=secs(1), id_type="patient")
    cat(sprintf("[%s] cnpt Größe: %s | Freier RAM danach: %.2f GB\n",
        parsed$short, format(object.size(cnpt), units="MB"), get_free_ram_gb()))

    cnpt$birthdate <- local_patient$birthdate[match(cnpt$subject_id, local_patient$subject_id)]
    cnpt$charttime_origin <- cnpt$birthdate + cnpt$charttime - 3600

    # RAM vor Tabelle laden
    cat(sprintf("[%s] Freier RAM vor Tabelle '%s': %.2f GB\n", parsed$short, parsed$table, get_free_ram_gb()))
    tbl <- miiv[[parsed$table]][, c("itemid", "subject_id", "valuenum")]
    cat(sprintf("[%s] tbl Größe: %s | Freier RAM danach: %.2f GB\n",
        parsed$short, format(object.size(tbl), units="MB"), get_free_ram_gb()))

    itemid_mask <- tbl[["itemid"]] %in% parsed$ids
    cnpt_from_event <- tbl[itemid_mask, ]
    rm(tbl)
    gc()

    if (!is.null(min_val) && !is.null(max_val)) {
        self_created_concept <- cnpt_from_event[!is.na(valuenum) & valuenum >= min_val & valuenum <= max_val]
    } else if (!is.null(min_val)) {
        self_created_concept <- cnpt_from_event[!is.na(valuenum) & valuenum >= min_val]
    } else if (!is.null(max_val)) {
        self_created_concept <- cnpt_from_event[!is.na(valuenum) & valuenum <= max_val]
    } else {
        self_created_concept <- cnpt_from_event[!is.na(valuenum)]
    }

    if (nrow(cnpt) == nrow(self_created_concept)) {
        print(paste(parsed$short, ": works"))
    } else {
        print(paste(parsed$short, ": NOT", nrow(cnpt), "vs", nrow(self_created_concept)))
    }

    rm(cnpt, cnpt_from_event, self_created_concept, itemid_mask)
    gc()
}

In [12]:
# %%R
# library(ricu)
# library(jsonlite)

# local_patient <- as.data.frame(miiv$patients[, c("subject_id", "anchor_year", "anchor_age")])
# local_patient$birthdate <- as.POSIXct(
#     paste0(local_patient$anchor_year - local_patient$anchor_age, "-01-01 00:00:00"),
#     format = "%Y-%m-%d %H:%M:%S", tz = "UTC"
# )

# for (values in found_data_info) {
#     cleaned <- gsub("'", '"', values)
#     cleaned <- gsub("None", "null", cleaned)
#     parsed  <- fromJSON(cleaned)
#     min_val <- if (!is.null(parsed$min)) as.numeric(parsed$min) else NULL
#     max_val <- if (!is.null(parsed$max)) as.numeric(parsed$max) else NULL

#     cnpt <- load_concepts(parsed$short, src="miiv", aggregate=FALSE, interval=secs(1), id_type="patient")
#     cnpt$birthdate <- local_patient$birthdate[match(cnpt$subject_id, local_patient$subject_id)]
#     cnpt$charttime_origin <- cnpt$birthdate + cnpt$charttime - 3600

#     # nur benötigte Spalten laden
#     tbl <- miiv[[parsed$table]][, c("itemid", "subject_id", "valuenum")]
#     itemid_mask <- tbl[["itemid"]] %in% parsed$ids
#     cnpt_from_event <- tbl[itemid_mask, ]
#     rm(tbl)  # sofort freigeben

#     if (!is.null(min_val) && !is.null(max_val)) {
#         self_created_concept <- cnpt_from_event[!is.na(valuenum) & valuenum >= min_val & valuenum <= max_val]
#     } else if (!is.null(min_val)) {
#         self_created_concept <- cnpt_from_event[!is.na(valuenum) & valuenum >= min_val]
#     } else if (!is.null(max_val)) {
#         self_created_concept <- cnpt_from_event[!is.na(valuenum) & valuenum <= max_val]
#     } else {
#         self_created_concept <- cnpt_from_event[!is.na(valuenum)]
#     }

#     if (nrow(cnpt) == nrow(self_created_concept)) {
#         print(paste(parsed$short, ": works"))
#     } else {
#         print(paste(parsed$short, ": NOT", nrow(cnpt), "vs", nrow(self_created_concept)))
#     }

#     rm(cnpt, cnpt_from_event, self_created_concept, itemid_mask)
#     gc()
# }

In [13]:
# %%R
# library(ricu)
# library(jsonlite)

# # einmal vor dem Loop berechnen
# local_patient <- as.data.frame(miiv$patients[, c("subject_id", "anchor_year", "anchor_age")])
# local_patient$birthdate <- as.POSIXct(
#     paste0(local_patient$anchor_year - local_patient$anchor_age, "-01-01 00:00:00"),
#     format = "%Y-%m-%d %H:%M:%S", tz = "UTC"
# )

# for (values in found_data_info) {
#     cleaned <- gsub("'", '"', values)
#     cleaned <- gsub("None", "null", cleaned)
#     parsed  <- fromJSON(cleaned)
#     min_val <- if (!is.null(parsed$min)) as.numeric(parsed$min) else NULL
#     max_val <- if (!is.null(parsed$max)) as.numeric(parsed$max) else NULL

#     cnpt <- load_concepts(parsed$short, src="miiv", aggregate=FALSE, interval=secs(1), id_type="patient")
#     cnpt$birthdate <- local_patient$birthdate[match(cnpt$subject_id, local_patient$subject_id)]
#     cnpt$charttime_origin <- cnpt$birthdate + cnpt$charttime - 3600

#     itemid_mask <- miiv[[parsed$table]][["itemid"]] %in% parsed$ids
#     cnpt_from_event <- miiv[[parsed$table]][itemid_mask, ]

#     if (!is.null(min_val) && !is.null(max_val)) {
#         self_created_concept <- cnpt_from_event[!is.na(valuenum) & valuenum >= min_val & valuenum <= max_val]
#     } else if (!is.null(min_val)) {
#         self_created_concept <- cnpt_from_event[!is.na(valuenum) & valuenum >= min_val]
#     } else if (!is.null(max_val)) {
#         self_created_concept <- cnpt_from_event[!is.na(valuenum) & valuenum <= max_val]
#     } else {
#         self_created_concept <- cnpt_from_event[!is.na(valuenum)]
#     }

#     if (nrow(cnpt) == nrow(self_created_concept)) {
#         print(paste(parsed$short, ": works"))
#     } else {
#         print(paste(parsed$short, ": NOT", nrow(cnpt), "vs", nrow(self_created_concept)))
#     }

#     # Speicher freigeben nach jedem Durchlauf
#     rm(cnpt, cnpt_from_event, self_created_concept, itemid_mask)
#     gc()
# }

In [14]:
# %%R
# library(ricu)
# library(jsonlite)

# local_patient <- as.data.frame(miiv$patients[, c("subject_id", "anchor_year", "anchor_age")])
# local_patient$birthdate <- as.POSIXct(paste0(local_patient$anchor_year - local_patient$anchor_age, "-01-01 00:00:00"), format="%Y-%m-%d %H:%M:%S", tz="UTC")

# for (values in found_data_info) {
#     cleaned <- gsub("'", '"', values)
#     cleaned <- gsub("None", "null", cleaned)
#     parsed  <- fromJSON(cleaned)

#     min_val <- if (!is.null(parsed$min)) as.numeric(parsed$min) else NULL
#     max_val <- if (!is.null(parsed$max)) as.numeric(parsed$max) else NULL

#     cnpt <- load_concepts(parsed$short, src="miiv", aggregate=FALSE, interval=secs(1), id_type="patient")
    
#     cnpt$birthdate <- local_patient$birthdate[match(cnpt$subject_id, local_patient$subject_id)]
#     cnpt$charttime_origin <- cnpt$birthdate + cnpt$charttime - 3600

#     itemid_mask <- miiv[[parsed$table]][["itemid"]] %in% parsed$ids
#     cnpt_from_event <- miiv[[parsed$table]][itemid_mask, ]

#     if (!is.null(min_val) && !is.null(max_val)) {
#         self_created_concept <- cnpt_from_event[!is.na(valuenum) & valuenum >= min_val & valuenum <= max_val]
#     } else if (!is.null(min_val)) {
#         self_created_concept <- cnpt_from_event[!is.na(valuenum) & valuenum >= min_val]
#     } else if (!is.null(max_val)) {
#         self_created_concept <- cnpt_from_event[!is.na(valuenum) & valuenum <= max_val]
#     } else {
#         self_created_concept <- cnpt_from_event[!is.na(valuenum)]
#     }

#     if (nrow(cnpt) == nrow(self_created_concept)) {
#         print(paste(parsed$short, ": works"))
#     } else {
#         print(paste(parsed$short, ": NOT", nrow(cnpt), "vs", nrow(self_created_concept)))
#     }
# }

In [15]:
# %%R

# library(ricu)
# library(jsonlite)

# for (values in found_data_info){
#     cleaned <- gsub("'", '"', values)
#     cleaned <- gsub("None", 'null', cleaned)
#     parsed  <- fromJSON(cleaned)

#     # cnpt
#     cnpt = load_concepts(parsed$short, src="miiv", aggregate = FALSE, interval = secs(1), id_type = "patient")
#     local_patient = as.data.frame(miiv$patients[, c("subject_id", "anchor_year", "anchor_age")])
#     local_patient$birthdate = as.POSIXct(paste0(local_patient$anchor_year - local_patient$anchor_age, "-01-01 00:00:00"), format = "%Y-%m-%d %H:%M:%S", tz = "UTC")
#     cnpt$birthdate = local_patient$birthdate[match(cnpt$subject_id, local_patient$subject_id)]
#     cnpt$charttime_origin = cnpt$birthdate + cnpt$charttime - 3600
#     cnpt

#     # cnpt_from_event
#     itemid_mask = miiv[[parsed$table]][["itemid"]] %in% parsed$ids
#     cnpt_from_event = miiv[[parsed$table]][itemid_mask, ]

#     # self_created_concept
#         print(parsed$min)
#         print(parsed$max)

#     if (!is.null(parsed$min) & !is.null(parsed$max)){
#         self_created_concept = cnpt_from_event[!is.na(valuenum) & valuenum >= parsed$min & valuenum <= parsed$max]
#     }else if (!is.null(parsed$min)){
#         self_created_concept = cnpt_from_event[!is.na(valuenum) & valuenum >= parsed$min]
#     }else if (!is.null(parsed$max)){
#         self_created_concept = cnpt_from_event[!is.na(valuenum) & valuenum <= parsed$max]
#     }else{
#         self_created_concept = cnpt_from_event[!is.na(valuenum)]
#     }

#     if (nrow(cnpt) == nrow(self_created_concept)){
#         print("works")
#     }else{
#         print("NOT")
#         print(nrow(cnpt))
#         print(nrow(self_created_concept))
#     }
# }
